# Import các thư viện vần thiết

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import save_npz
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel, VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
import joblib
import os

# Đọc dữ liệu đã qua tiền xử lý

In [2]:
DATA_PATH = "../data/"

required_files = ["train_preprocessed.csv", "test_preprocessed.csv"]
missing_files = [f for f in required_files if not os.path.exists(DATA_PATH + "processed/" + f)]
if missing_files:
    raise FileNotFoundError(
        "Missing preprocessing files: " + ", ".join(missing_files)
    )

train_preprocessed = pd.read_csv(DATA_PATH + "processed/train_preprocessed.csv")
test_preprocessed = pd.read_csv(DATA_PATH + "processed/test_preprocessed.csv")

print("Train preprocessed shape:", train_preprocessed.shape)
print("Test preprocessed shape:", test_preprocessed.shape)
train_preprocessed.head()

Train preprocessed shape: (140700, 25)
Test preprocessed shape: (93800, 24)


,id,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,...,Work/Study Hours,Financial Stress,Family History of Mental Illness,Academic Pressure Missing Relevant,Study Satisfaction Missing Relevant,CGPA Missing Relevant,Work Pressure Missing Relevant,Job Satisfaction Missing Relevant,Profession Missing Relevant,Depression
0,0,Female,49.0,Ludhiana,Working Professional,Chef,0.0,5.0,0.00,0.0,...,1.0,2.0,No,0.0,0.0,0.0,0.0,0.0,0.0,0
1,1,Male,26.0,Varanasi,Working Professional,Teacher,0.0,4.0,0.00,0.0,...,7.0,3.0,No,0.0,0.0,0.0,0.0,0.0,0.0,1
2,2,Male,33.0,Visakhapatnam,Student,Not applicable,5.0,0.0,8.97,2.0,...,3.0,1.0,No,0.0,0.0,0.0,0.0,0.0,0.0,1
3,3,Male,22.0,Mumbai,Working Professional,Teacher,0.0,5.0,0.00,0.0,...,10.0,1.0,Yes,0.0,0.0,0.0,0.0,0.0,0.0,1
4,4,Female,30.0,Kanpur,Working Professional,Business Analyst,0.0,1.0,0.00,0.0,...,9.0,4.0,Yes,0.0,0.0,0.0,0.0,0.0,0.0,0


## Feature Engineering Pipeline (Extract -> Transform -> Selection)

1. Nạp dữ liệu đã clean + impute từ preprocessing.
2. Feature extraction theo domain (sleep, diet, pressure, risk indices).
3. Feature transform: impute/scaler cho numeric, one-hot cho categorical.
4. Feature selection: lọc biến hằng + chọn biến quan trọng bằng L1.
5. Tạo dữ liệu sẵn sàng training và huấn luyện baseline model.
6. Lưu artifacts để tái sử dụng.

In [3]:
TARGET_COL = "Depression"
ID_COL = "id"
df_fe = train_preprocessed.copy()

## Đồng Bộ Các Trường Cần Cho Feature Mới

Sau khi dữ liệu đã được clean + impute thì chuẩn hóa để trích xuất feature nhất quán.

Ánh xạ nhị phân:

$$
\text{bin}(x)=
\begin{cases}
1, & x=\text{Yes}\\
0, & x=\text{No}\\
\text{NaN}, & \text{khác/thiếu}
\end{cases}
$$

Tạo biến `is_student` để dùng cho các biến hợp nhất theo vai trò.

In [4]:
binary_map = {"Yes": 1, "No": 0}

df_fe["suicidal_thoughts"] = df_fe["Have you ever had suicidal thoughts ?"].map(binary_map)
df_fe["family_history_mi"] = df_fe["Family History of Mental Illness"].map(binary_map)

role_col = "Working Professional or Student"
df_fe["is_student"] = df_fe[role_col].astype(str).str.lower().eq("student").astype(int)

df_fe[["suicidal_thoughts", "family_history_mi", "is_student"]].head()

,suicidal_thoughts,family_history_mi,is_student
0,0,0,0
1,1,0,0
2,1,0,1
3,1,1,0
4,1,1,0


## Hợp Nhất Biến Theo Vai Trò (Student vs Working Professional)

Lý do:
- `Academic Pressure` và `Work Pressure` mô tả cùng một khái niệm áp lực nhưng ở 2 nhóm khác nhau.
- Tương tự với `Study Satisfaction` và `Job Satisfaction`.

Ta xây dựng biến hợp nhất:

$$
\text{primary\_pressure}_i =
\begin{cases}
\text{AcademicPressure}_i, & \text{nếu Student}\\
\text{WorkPressure}_i, & \text{nếu Working Professional}
\end{cases}
$$

$$
\text{primary\_satisfaction}_i =
\begin{cases}
\text{StudySatisfaction}_i, & \text{nếu Student}\\
\text{JobSatisfaction}_i, & \text{nếu Working Professional}
\end{cases}
$$

Cách này giúp mô hình nhìn thấy tín hiệu nhất quán trên toàn bộ dân số thay vì tách rời theo subgroup.

In [5]:
# Cast numeric columns
numeric_candidates = [
    "Age", "Academic Pressure", "Work Pressure", "CGPA", "Study Satisfaction",
    "Job Satisfaction", "Work/Study Hours", "Financial Stress"
]
for col in numeric_candidates:
    if col in df_fe.columns:
        df_fe[col] = pd.to_numeric(df_fe[col], errors="coerce")

# Merge features based on role
df_fe["primary_pressure"] = np.where(
    df_fe["is_student"].eq(1),
    df_fe["Academic Pressure"],
    df_fe["Work Pressure"]
)

df_fe["primary_satisfaction"] = np.where(
    df_fe["is_student"].eq(1),
    df_fe["Study Satisfaction"],
    df_fe["Job Satisfaction"]
)

df_fe["pressure_missing_flag"] = df_fe["primary_pressure"].isna().astype(int)
df_fe["satisfaction_missing_flag"] = df_fe["primary_satisfaction"].isna().astype(int)

df_fe[["is_student", "primary_pressure", "primary_satisfaction"]].head()

,is_student,primary_pressure,primary_satisfaction
0,0,5.0,2.0
1,0,4.0,3.0
2,1,5.0,2.0
3,0,5.0,1.0
4,0,1.0,1.0


## Lượng Hóa Giấc Ngủ Và Xử Lý Unknown

Lý do:
- `Sleep Duration` sau preprocessing có thể còn mức `Unknown` (giá trị không xác định hoặc không parse được).
- Khi đổi sang biến số, cần giữ lại thông tin `Unknown` thay vì làm mất tín hiệu.

Ánh xạ giờ ngủ (xấp xỉ trung điểm khoảng):
- `Less than 5 hours` $\rightarrow 4.5$
- `5-6 hours` $\rightarrow 5.5$
- `7-8 hours` $\rightarrow 7.5$
- `More than 8 hours` $\rightarrow 8.5$

Đặc trưng tạo thêm:

$$
\text{sleep\_unknown\_flag}=\mathbb{1}(\text{SleepDuration}=\text{Unknown})
$$

$$
\text{sleep\_deviation} = |\text{sleep\_hours} - 7.5|,
\quad
\text{sleep\_debt} = \max(0, 7 - \text{sleep\_hours})
$$

Giá trị không ánh xạ được giữ ở `NaN` để xử lý bằng imputer ở bước modeling.

In [6]:
sleep_map = {
    "Less than 5 hours": 4.5,
    "5-6 hours": 5.5,
    "7-8 hours": 7.5,
    "More than 8 hours": 8.5
}

df_fe["sleep_unknown_flag"] = df_fe["Sleep Duration"].astype(str).eq("Unknown").astype(int)
df_fe["sleep_hours"] = df_fe["Sleep Duration"].map(sleep_map)
df_fe["sleep_hours_missing_flag"] = df_fe["sleep_hours"].isna().astype(int)
df_fe["sleep_deviation"] = (df_fe["sleep_hours"] - 7.5).abs()
df_fe["sleep_debt"] = np.maximum(0, 7 - df_fe["sleep_hours"])

df_fe[[
    "Sleep Duration", "sleep_unknown_flag", "sleep_hours",
    "sleep_hours_missing_flag", "sleep_deviation", "sleep_debt"
]].head()

,Sleep Duration,sleep_unknown_flag,sleep_hours,sleep_hours_missing_flag,sleep_deviation,sleep_debt
0,More than 8 hours,0,8.5,0,1.0,0.0
1,Less than 5 hours,0,4.5,0,3.0,2.5
2,5-6 hours,0,5.5,0,2.0,1.5
3,Less than 5 hours,0,4.5,0,3.0,2.5
4,5-6 hours,0,5.5,0,2.0,1.5


## Tạo Đặc Trưng Từ Diet, Age, Work/Study Hours

Lý do:
- Nhiều quan hệ trong sức khỏe tâm thần không tuyến tính.
- `Dietary Habits` có thể chứa `Unknown`, cần giữ tín hiệu này qua một biến cờ.

Thiết kế đặc trưng:

1) Điểm chế độ ăn:

$$
\text{diet\_score} \in \{0,1,2\},\quad
\text{Unhealthy}=0,\ \text{Moderate}=1,\ \text{Healthy}=2
$$

2) Cờ giá trị chưa xác định:

$$
\text{diet\_unknown\_flag}=\mathbb{1}(\text{DietaryHabits}=\text{Unknown})
$$

3) Thành phần phi tuyến giờ học/làm:

$$
\text{hours\_sq} = (\text{work\_study\_hours})^2
$$

4) Cờ quá tải thời gian:

$$
\text{overwork\_flag}=\mathbb{1}(\text{work\_study\_hours} > 8)
$$

5) Nhóm tuổi (life-stage) để mô hình học khác biệt theo giai đoạn sống.

In [7]:
diet_map = {"Unhealthy": 0, "Moderate": 1, "Healthy": 2}

df_fe["diet_unknown_flag"] = df_fe["Dietary Habits"].astype(str).eq("Unknown").astype(int)
df_fe["diet_score"] = df_fe["Dietary Habits"].map(diet_map)
df_fe["diet_score_missing_flag"] = df_fe["diet_score"].isna().astype(int)

df_fe["work_study_hours"] = pd.to_numeric(df_fe["Work/Study Hours"], errors="coerce")
df_fe["work_hours_missing_flag"] = df_fe["work_study_hours"].isna().astype(int)
df_fe["hours_sq"] = df_fe["work_study_hours"] ** 2
df_fe["overwork_flag"] = df_fe["work_study_hours"].gt(8).astype(int)

df_fe["age_group"] = pd.cut(
    df_fe["Age"],
    bins=[0, 18, 25, 35, 50, 120],
    labels=["teen", "young_adult", "adult", "mid_age", "senior"],
    right=False,
    include_lowest=True
)

df_fe[[
    "Dietary Habits", "diet_unknown_flag", "diet_score", "diet_score_missing_flag",
    "work_study_hours", "work_hours_missing_flag", "hours_sq", "overwork_flag", "age_group"
]].head()

,Dietary Habits,diet_unknown_flag,diet_score,diet_score_missing_flag,work_study_hours,work_hours_missing_flag,hours_sq,overwork_flag,age_group
0,Healthy,0,2.0,0,1.0,0,1.0,0,mid_age
1,Unhealthy,0,0.0,0,7.0,0,49.0,0,adult
2,Healthy,0,2.0,0,3.0,0,9.0,0,adult
3,Moderate,0,1.0,0,10.0,0,100.0,1,young_adult
4,Unhealthy,0,0.0,0,9.0,0,81.0,1,adult


## Xây Dựng Chỉ Số Tương Tác Rủi Ro (Risk Indices)

Lý do:
- Trầm cảm thường do cộng hưởng nhiều yếu tố thay vì một biến đơn lẻ.
- Chỉ số tổ hợp giúp mô hình nhận diện hiệu ứng cộng dồn.

Định nghĩa các chỉ số:

1) Gánh nặng stress:

$$
\text{stress\_load} = 0.35\,\text{primary\_pressure} + 0.35\,\text{financial\_stress} + 0.20\,\text{suicidal\_thoughts} + 0.10\,\text{family\_history}
$$

2) Rủi ro lối sống:

$$
\text{lifestyle\_risk} = 0.50\,\text{sleep\_debt} + 0.30\,(2-\text{diet\_score}) + 0.20\,\text{overwork\_flag}
$$

3) Rủi ro ròng có xét yếu tố bảo vệ:

$$
\text{net\_risk} = \text{stress\_load} + \text{lifestyle\_risk} - 0.25\,\text{primary\_satisfaction}
$$

Trọng số ở đây là heuristic có định hướng miền kiến thức; bạn có thể fine-tune bằng validation.

In [8]:
# Cast engineered features to numeric and fill missing values
for col in ["primary_pressure", "Financial Stress", "suicidal_thoughts", "family_history_mi", "sleep_debt", "diet_score", "overwork_flag", "primary_satisfaction"]:
    if col in df_fe.columns:
        df_fe[col] = pd.to_numeric(df_fe[col], errors="coerce")

num_for_index = [
    "primary_pressure", "Financial Stress", "suicidal_thoughts", "family_history_mi",
    "sleep_debt", "diet_score", "overwork_flag", "primary_satisfaction"
]
for col in num_for_index:
    median_val = df_fe[col].median()
    df_fe[col] = df_fe[col].fillna(median_val)

df_fe["stress_load"] = (
    0.35 * df_fe["primary_pressure"]
    + 0.35 * df_fe["Financial Stress"]
    + 0.20 * df_fe["suicidal_thoughts"]
    + 0.10 * df_fe["family_history_mi"]
)

df_fe["lifestyle_risk"] = (
    0.50 * df_fe["sleep_debt"]
    + 0.30 * (2 - df_fe["diet_score"])
    + 0.20 * df_fe["overwork_flag"]
)

df_fe["net_risk"] = df_fe["stress_load"] + df_fe["lifestyle_risk"] - 0.25 * df_fe["primary_satisfaction"]

df_fe[["stress_load", "lifestyle_risk", "net_risk"]].describe().T

,count,mean,std,min,25%,50%,75%,max
stress_load,140700.0,2.254197,0.735805,0.55,1.75,2.2,2.75,3.80
lifestyle_risk,140700.0,0.892223,0.603276,0.00,0.30,0.8,1.35,2.05
net_risk,140700.0,2.404355,1.056262,-0.55,1.65,2.4,3.15,5.75


## Feature Transform + Selection + Baseline Model

Sau khi extraction, ta chuyển sang pipeline cho training:

1. Transform:
- Numeric: `median imputation + RobustScaler`.
- Categorical: `most_frequent imputation + OneHotEncoder`.

2. Selection:
- `VarianceThreshold` loại feature hằng.
- `SelectFromModel(LogisticRegression L1)` giữ feature có tín hiệu.

3. Training:
- LogisticRegression `class_weight='balanced'` để baseline cho bài toán mất cân bằng lớp.

In [9]:
# Prepare data for modeling (baseline)
X = df_fe.drop(columns=[c for c in [TARGET_COL, ID_COL] if c in df_fe.columns])
y = df_fe[TARGET_COL].astype(int)

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

feature_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("variance", VarianceThreshold(threshold=0.0)),
    (
        "selector",
        SelectFromModel(
            LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=0.5,
                max_iter=2000,
                class_weight="balanced"
            ),
            threshold="median"
        )
    )
])

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_ready = feature_pipeline.fit_transform(X_train, y_train)
X_valid_ready = feature_pipeline.transform(X_valid)

clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit(X_train_ready, y_train)

y_proba = clf.predict_proba(X_valid_ready)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

all_feature_names = np.array(feature_pipeline.named_steps["preprocessor"].get_feature_names_out())
var_mask = feature_pipeline.named_steps["variance"].get_support()
post_var_names = all_feature_names[var_mask]
sel_mask = feature_pipeline.named_steps["selector"].get_support()
selected_feature_names = post_var_names[sel_mask]

print(f"Raw features before transform: {X.shape[1]}")
print(f"Features after variance filter: {len(post_var_names)}")
print(f"Selected features: {len(selected_feature_names)}")
print(f"ROC-AUC: {roc_auc_score(y_valid, y_proba):.4f}")
print(f"F1-score: {f1_score(y_valid, y_pred):.4f}")
print("\nClassification report:\n")
print(classification_report(y_valid, y_pred))

Raw features before transform: 46
Features after variance filter: 152
Selected features: 76
ROC-AUC: 0.9738
F1-score: 0.7994

Classification report:

              precision    recall  f1-score   support

           0       0.98      0.91      0.95     23027
           1       0.70      0.93      0.80      5113

    accuracy                           0.92     28140
   macro avg       0.84      0.92      0.87     28140
weighted avg       0.93      0.92      0.92     28140



In [10]:
def engineer_features(df_input: pd.DataFrame, fill_values: dict | None = None) -> pd.DataFrame:
    df_out = df_input.copy()

    binary_map = {"Yes": 1, "No": 0}
    if "Have you ever had suicidal thoughts ?" in df_out.columns:
        df_out["suicidal_thoughts"] = df_out["Have you ever had suicidal thoughts ?"].map(binary_map)
    if "Family History of Mental Illness" in df_out.columns:
        df_out["family_history_mi"] = df_out["Family History of Mental Illness"].map(binary_map)

    role_col = "Working Professional or Student"
    if role_col in df_out.columns:
        df_out["is_student"] = df_out[role_col].astype(str).str.lower().eq("student").astype(int)
    else:
        df_out["is_student"] = 0

    numeric_candidates = [
        "Age", "Academic Pressure", "Work Pressure", "CGPA", "Study Satisfaction",
        "Job Satisfaction", "Work/Study Hours", "Financial Stress"
    ]
    for col in numeric_candidates:
        if col in df_out.columns:
            df_out[col] = pd.to_numeric(df_out[col], errors="coerce")

    if "Academic Pressure" in df_out.columns and "Work Pressure" in df_out.columns:
        df_out["primary_pressure"] = np.where(
            df_out["is_student"].eq(1),
            df_out["Academic Pressure"],
            df_out["Work Pressure"]
        )
    else:
        df_out["primary_pressure"] = np.nan

    if "Study Satisfaction" in df_out.columns and "Job Satisfaction" in df_out.columns:
        df_out["primary_satisfaction"] = np.where(
            df_out["is_student"].eq(1),
            df_out["Study Satisfaction"],
            df_out["Job Satisfaction"]
        )
    else:
        df_out["primary_satisfaction"] = np.nan

    df_out["pressure_missing_flag"] = df_out.get("Academic Pressure Missing Relevant", 0)
    df_out["satisfaction_missing_flag"] = df_out.get("Study Satisfaction Missing Relevant", 0)

    sleep_map = {
        "Less than 5 hours": 4.5,
        "5-6 hours": 5.5,
        "7-8 hours": 7.5,
        "More than 8 hours": 8.5
    }
    if "Sleep Duration" in df_out.columns:
        df_out["sleep_unknown_flag"] = df_out["Sleep Duration"].astype(str).eq("Unknown").astype(int)
        df_out["sleep_hours"] = df_out["Sleep Duration"].map(sleep_map)
        df_out["sleep_hours_missing_flag"] = df_out["sleep_hours"].isna().astype(int)
        df_out["sleep_deviation"] = (df_out["sleep_hours"] - 7.5).abs()
        df_out["sleep_debt"] = np.maximum(0, 7 - df_out["sleep_hours"])
    else:
        df_out["sleep_unknown_flag"] = 0
        df_out["sleep_hours"] = np.nan
        df_out["sleep_hours_missing_flag"] = 1
        df_out["sleep_deviation"] = np.nan
        df_out["sleep_debt"] = np.nan

    diet_map = {"Unhealthy": 0, "Moderate": 1, "Healthy": 2}
    if "Dietary Habits" in df_out.columns:
        df_out["diet_unknown_flag"] = df_out["Dietary Habits"].astype(str).eq("Unknown").astype(int)
        df_out["diet_score"] = df_out["Dietary Habits"].map(diet_map)
        df_out["diet_score_missing_flag"] = df_out["diet_score"].isna().astype(int)
    else:
        df_out["diet_unknown_flag"] = 0
        df_out["diet_score"] = np.nan
        df_out["diet_score_missing_flag"] = 1

    if "Work/Study Hours" in df_out.columns:
        df_out["work_study_hours"] = pd.to_numeric(df_out["Work/Study Hours"], errors="coerce")
        df_out["work_hours_missing_flag"] = df_out["work_study_hours"].isna().astype(int)
        df_out["hours_sq"] = df_out["work_study_hours"] ** 2
        df_out["overwork_flag"] = df_out["work_study_hours"].gt(8).astype(int)
    else:
        df_out["work_study_hours"] = np.nan
        df_out["work_hours_missing_flag"] = 1
        df_out["hours_sq"] = np.nan
        df_out["overwork_flag"] = np.nan

    if "Age" in df_out.columns:
        df_out["age_group"] = pd.cut(
            df_out["Age"],
            bins=[0, 18, 25, 35, 50, 120],
            labels=["teen", "young_adult", "adult", "mid_age", "senior"],
            right=False,
            include_lowest=True
        )
    else:
        df_out["age_group"] = np.nan

    risk_cols = [
        "primary_pressure", "Financial Stress", "suicidal_thoughts", "family_history_mi",
        "sleep_debt", "diet_score", "overwork_flag", "primary_satisfaction"
    ]
    if fill_values is None:
        fill_values = {}
        for col in risk_cols:
            if col in df_out.columns:
                fill_values[col] = pd.to_numeric(df_out[col], errors="coerce").median()

    for col in risk_cols:
        if col in df_out.columns:
            df_out[col] = pd.to_numeric(df_out[col], errors="coerce")
            df_out[f"{col}_filled"] = df_out[col].fillna(fill_values.get(col, df_out[col].median()))
        else:
            df_out[col] = np.nan
            df_out[f"{col}_filled"] = fill_values.get(col, 0)

    df_out["stress_load"] = (
        0.35 * df_out["primary_pressure_filled"]
        + 0.35 * df_out["Financial Stress_filled"]
        + 0.20 * df_out["suicidal_thoughts_filled"]
        + 0.10 * df_out["family_history_mi_filled"]
    )

    df_out["lifestyle_risk"] = (
        0.50 * df_out["sleep_debt_filled"]
        + 0.30 * (2 - df_out["diet_score_filled"])
        + 0.20 * df_out["overwork_flag_filled"]
    )

    df_out["net_risk"] = (
        df_out["stress_load"] + df_out["lifestyle_risk"] - 0.25 * df_out["primary_satisfaction_filled"]
    )

    return df_out

# Build test features using train medians to avoid leakage
train_fill_values = {
    "primary_pressure": df_fe["primary_pressure"].median(),
    "Financial Stress": df_fe["Financial Stress"].median(),
    "suicidal_thoughts": df_fe["suicidal_thoughts"].median(),
    "family_history_mi": df_fe["family_history_mi"].median(),
    "sleep_debt": df_fe["sleep_debt"].median(),
    "diet_score": df_fe["diet_score"].median(),
    "overwork_flag": df_fe["overwork_flag"].median(),
    "primary_satisfaction": df_fe["primary_satisfaction"].median()
}

df_fe_test = engineer_features(test_preprocessed, fill_values=train_fill_values)

df_fe_test.head()

,id,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,...,Financial Stress_filled,suicidal_thoughts_filled,family_history_mi_filled,sleep_debt_filled,diet_score_filled,overwork_flag_filled,primary_satisfaction_filled,stress_load,lifestyle_risk,net_risk
0,140700,Male,53.0,Visakhapatnam,Working Professional,Judge,0.0,2.0,0.00,0.0,...,3.0,0,1,2.5,1.0,1,5.0,1.85,1.75,2.35
1,140701,Female,58.0,Kolkata,Working Professional,Educational Consultant,0.0,2.0,0.00,0.0,...,4.0,0,0,2.5,1.0,0,4.0,2.10,1.55,2.65
2,140702,Male,53.0,Jaipur,Working Professional,Teacher,0.0,4.0,0.00,0.0,...,4.0,1,0,0.0,1.0,1,1.0,3.00,0.50,3.25
3,140703,Female,23.0,Rajkot,Student,Not applicable,5.0,0.0,6.84,1.0,...,4.0,1,0,0.0,1.0,1,1.0,3.35,0.50,3.60
4,140704,Male,47.0,Kalyan,Working Professional,Teacher,0.0,5.0,0.00,0.0,...,4.0,1,0,0.0,1.0,0,5.0,3.35,0.30,2.40


## Lưu Feature Dataset Và Baseline Artifact

In [11]:
os.makedirs("../artifacts", exist_ok=True)
os.makedirs("../data/train", exist_ok=True)
os.makedirs("../data/test", exist_ok=True)

# Transform full training data for saving
X_ready_full = feature_pipeline.transform(X)

# Transform test data
X_test = df_fe_test.drop(columns=[c for c in [ID_COL, TARGET_COL] if c in df_fe_test.columns])
X_test_ready = feature_pipeline.transform(X_test)

# Save transformed data and feature names for modeling
save_npz(DATA_PATH + "train/train_X.npz", X_ready_full)
save_npz(DATA_PATH + "test/test_X.npz", X_test_ready)

pd.Series(selected_feature_names, name="feature_name").to_csv(
    DATA_PATH + "train/train_feature_names.csv", index=False
)

# Save target variable for modeling
y.to_csv(DATA_PATH + "train/train_y.csv", index=False)

# Save engineered feature tables for reference
feature_table = df_fe.copy()
feature_table.to_csv(DATA_PATH + "train/train_feature_engineered.csv", index=False)

test_feature_table = df_fe_test.copy()
test_feature_table.to_csv(DATA_PATH + "test/test_feature_engineered.csv", index=False)

# Save objects to reuse in modeling
joblib.dump(feature_pipeline, "../artifacts/feature_pipeline.joblib")
joblib.dump(clf, "../artifacts/feature_model_baseline.joblib")

print("Saved: ../data/train_feature_engineered.csv")
print("Saved: ../data/test_feature_engineered.csv")
print("Saved: ../data/train_X.npz")
print("Saved: ../data/test_X.npz")
print("Saved: ../data/train_feature_names.csv")
print("Saved: ../data/train_y.csv")
print("Saved: ../artifacts/feature_pipeline.joblib")
print("Saved: ../artifacts/feature_model_baseline.joblib")
print("Final engineered train table shape:", feature_table.shape)
print("Final engineered test table shape:", test_feature_table.shape)
print("Train-ready matrix shape:", X_ready_full.shape)
print("Test-ready matrix shape:", X_test_ready.shape)

Saved: ../data/train_feature_engineered.csv
Saved: ../data/test_feature_engineered.csv
Saved: ../data/train_X.npz
Saved: ../data/test_X.npz
Saved: ../data/train_feature_names.csv
Saved: ../data/train_y.csv
Saved: ../artifacts/feature_pipeline.joblib
Saved: ../artifacts/feature_model_baseline.joblib
Final engineered train table shape: (140700, 48)
Final engineered test table shape: (93800, 55)
Train-ready matrix shape: (140700, 76)
Test-ready matrix shape: (93800, 76)
